# DZ 24 — Полноценный ML-сервис
Ноутбук: обучение, логирование в MLflow, регистрация модели `diabets`, загрузка версии модели и сохранение локально для FastAPI.


In [ ]:
import os
import joblib
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.datasets import load_diabetes
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score


In [ ]:
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://127.0.0.1:5000')
EXPERIMENT_NAME = 'diabets-experiment'
REGISTERED_MODEL_NAME = 'diabets'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
client = MlflowClient()


In [ ]:
X, y = load_diabetes(return_X_y=True, scaled=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)),
])
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
mae, r2


In [ ]:
with mlflow.start_run(run_name='random-forest-diabetes') as run:
    mlflow.log_param('model', 'RandomForestRegressor')
    mlflow.log_param('n_estimators', 200)
    mlflow.log_param('max_depth', 10)
    mlflow.log_metric('mae', mae)
    mlflow.log_metric('r2', r2)
    mlflow.sklearn.log_model(pipeline, artifact_path='model', registered_model_name=REGISTERED_MODEL_NAME)
    run_id = run.info.run_id
run_id


In [ ]:
versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
latest_version = sorted(versions, key=lambda x: int(x.version))[-1]
model_uri = f'models:/{REGISTERED_MODEL_NAME}/{latest_version.version}'
loaded_model = mlflow.sklearn.load_model(model_uri)
loaded_model.predict(X_test[:3])


In [ ]:
os.makedirs('../model', exist_ok=True)
joblib.dump(loaded_model, '../model/diabets_model.joblib')
